# Construção de um modelo preditivo em dados massivos com Spark ML

## Regressão — previsão da duração de viagens de bicicleta compartilhada (Divvy / Chicago)

**Disciplina:** Processamento de Dados Massivos — Prof. Alexandre Roriz

**Tipo de problema (sorteado):** Regressão

Este notebook percorre todo o pipeline de ciência de dados sobre uma base massiva usando **Apache Spark (PySpark)**: ingestão com schema definido manualmente, exploração e limpeza, engenharia de atributos organizada em um `Pipeline`, modelagem com `LinearRegression` (incluindo ajuste de hiperparâmetros sem vazamento de dados), avaliação com as métricas de regressão (RMSE, MAE, R²) comparadas a um baseline, persistência do modelo com `save()` e conclusão.

Ele foi escrito para rodar no **Google Colab**, que já traz o PySpark instalado. Com `master("local[*]")`, o Spark usa todos os núcleos disponíveis como executores, simulando localmente o modelo de execução distribuída de um cluster.

## 1. Contextualização

### Base de dados

- **Fonte:** *Divvy Bike-Share System Data*, o programa público de bicicletas compartilhadas da cidade de Chicago, operado pela Lyft. Os dados são abertos e publicados mensalmente.
- **Link de acesso (reproduzível):** <https://divvy-tripdata.s3.amazonaws.com/index.html> — os arquivos mensais ficam em `https://divvy-tripdata.s3.amazonaws.com/AAAAMM-divvy-tripdata.zip` (download direto, sem autenticação).
- **Página oficial do programa:** <https://divvybikes.com/system-data>
- **Período usado neste notebook:** abril, maio e junho de **2024** (3 arquivos mensais).
- **Volume:** cada mês tem da ordem de 400 mil a 700 mil viagens. Só abril/2024 já possui **415.025** registros — acima do mínimo de 200 mil exigido. Os três meses juntos ultrapassam **1,6 milhão** de viagens.
- **Colunas (13):** `ride_id`, `rideable_type`, `started_at`, `ended_at`, `start_station_name`, `start_station_id`, `end_station_name`, `end_station_id`, `start_lat`, `start_lng`, `end_lat`, `end_lng`, `member_casual`.

### Pergunta de negócio

> **Dadas as características de uma viagem (tipo de bicicleta, horário, dia da semana, distância entre as estações e tipo de usuário), quanto tempo — em minutos — essa viagem tende a durar?**

Prever a duração das viagens ajuda o operador no **rebalanceamento da frota** (antecipar quando e onde bicicletas ficarão indisponíveis), no **dimensionamento das estações** e na **estimativa de receita** (planos casuais são cobrados por tempo de uso).

### Por que é um problema de dados massivos

Não se trata apenas de um arquivo grande:

- A Divvy registra **5 a 6 milhões de viagens por ano**, e a série histórica aberta vai de 2013 até hoje, somando **dezenas de milhões de registros** e centenas de GB. Esse volume não cabe na memória de uma única máquina, o que exige **processamento distribuído** — leitura particionada, transformações e agregações executadas em paralelo entre vários nós, exatamente o modelo do Spark.
- O caso é de **ingestão contínua** (novos arquivos todo mês), favorecendo pipelines que escalam horizontalmente em vez de reprocessamentos monolíticos.
- As operações do fluxo (limpeza, *joins* de estações, cálculo de distância, treino de modelos) são naturalmente paralelizáveis — o Spark as distribui pelos executores sem alterar o código da lógica.

### Uso de amostra

Para que o notebook rode em tempo hábil no Colab, **após a limpeza** tiramos uma amostra aleatória do conjunto dos três meses. Isso está explicitamente declarado: o **conjunto original** carregado tem mais de 1,6 milhão de viagens; a amostra de trabalho tem `N_AMOSTRA` linhas (abaixo), extraída por amostragem aleatória simples com semente fixa (`SEED`) para ser reprodutível. Em um cluster real, o mesmo código roda sobre a base completa apenas trocando o caminho e o `master`.

### 0. Parâmetros do experimento

Centraliza o que pode ser alterado: meses usados, tamanho da amostra e semente. Todo o restante do notebook usa estas variáveis.

In [ ]:
ANO = 2024
MESES = [4, 5, 6]            # abril, maio e junho de 2024
N_AMOSTRA = 300_000          # tamanho da amostra de trabalho (> 200 mil exigidos)
SEED = 42

BASE = "/content"
DIR_DADOS = f"{BASE}/dados/divvy"       # CSVs mensais baixados da fonte
DIR_MODELO = f"{BASE}/modelo_duracao"   # pipeline treinado persistido com save()

URL_DIVVY = "https://divvy-tripdata.s3.amazonaws.com"


def nome_zip(ano, mes):
    return f"{ano}{mes:02d}-divvy-tripdata.zip"


def nome_csv(ano, mes):
    return f"{ano}{mes:02d}-divvy-tripdata.csv"

## 2. Ingestão

### Download da fonte original

Conforme o enunciado, **o notebook baixa a base a partir da fonte** (nenhum dado é versionado no GitHub). Cada mês é um `.zip` com um `.csv` dentro. O `-nc` do `wget` evita rebaixar o que já existe.

In [ ]:
import os
import zipfile

os.makedirs(DIR_DADOS, exist_ok=True)

for mes in MESES:
    zip_path = f"{DIR_DADOS}/{nome_zip(ANO, mes)}"
    csv_path = f"{DIR_DADOS}/{nome_csv(ANO, mes)}"
    if not os.path.exists(csv_path):
        !wget -q -nc -P {DIR_DADOS} {URL_DIVVY}/{nome_zip(ANO, mes)}
        with zipfile.ZipFile(zip_path) as z:
            # extrai apenas o CSV principal (ignora pastas __MACOSX etc.)
            for membro in z.namelist():
                if membro.endswith(".csv") and not membro.startswith("__MACOSX"):
                    z.extract(membro, DIR_DADOS)

!ls -lh {DIR_DADOS}/*.csv

### Sessão Spark

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("Regressao_Divvy_DuracaoViagem")
         .master("local[*]")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.driver.memory", "6g")
         .getOrCreate())

print("Versão do Spark:", spark.version)
print("Paralelismo (núcleos):", spark.sparkContext.defaultParallelism)

### Leitura com schema definido manualmente (`StructType`)

Em vez de `inferSchema`, o schema é declarado explicitamente. Isso é a prática recomendada em dados massivos porque:

- **evita um passo extra de varredura** dos dados só para descobrir os tipos (`inferSchema` lê o arquivo duas vezes);
- **garante tipos estáveis** entre os meses — os timestamps são lidos como `timestamp` e as coordenadas como `double`, sem depender de heurística;
- torna a leitura **determinística e reproduzível**.

O formato de data dos arquivos Divvy é `yyyy-MM-dd HH:mm:ss`, informado em `timestampFormat`.

In [ ]:
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, TimestampType)

schema = StructType([
    StructField("ride_id", StringType(), False),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", DoubleType(), True),
    StructField("start_lng", DoubleType(), True),
    StructField("end_lat", DoubleType(), True),
    StructField("end_lng", DoubleType(), True),
    StructField("member_casual", StringType(), True),
])

caminhos = [f"{DIR_DADOS}/{nome_csv(ANO, m)}" for m in MESES]

bruto = (spark.read
         .option("header", True)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
         .schema(schema)
         .csv(caminhos))

print("Conjunto ORIGINAL carregado (3 meses):", f"{bruto.count():,} viagens")
bruto.printSchema()
bruto.show(5, truncate=False)

## 3. Exploração e limpeza

Toda a exploração e limpeza é feita **com funções do Spark**, sem trazer os dados para o driver.

### Estatísticas descritivas iniciais

In [ ]:
# Visão geral das colunas numéricas (coordenadas) do conjunto bruto
bruto.select("start_lat", "start_lng", "end_lat", "end_lng").describe().show()

# Distribuição das categóricas
print("rideable_type:")
bruto.groupBy("rideable_type").count().orderBy("count", ascending=False).show()
print("member_casual:")
bruto.groupBy("member_casual").count().orderBy("count", ascending=False).show()

### Contagem de valores nulos por coluna

As viagens cujo destino não foi registrado (dock desconhecido) trazem `end_station_*` e, em alguns casos, `end_lat/end_lng` nulos. Como a distância entre origem e destino é um atributo central, as linhas sem coordenadas de destino precisam ser descartadas.

In [ ]:
from pyspark.sql.functions import col, count, when

nulos = bruto.select([
    count(when(col(c).isNull(), c)).alias(c) for c in bruto.columns
])
nulos.show(truncate=False)

### Duplicatas

`ride_id` é o identificador único de cada viagem. Removemos duplicatas por garantia (reprocessamentos ou sobreposição entre arquivos mensais podem gerá-las).

In [ ]:
total = bruto.count()
distintos = bruto.select("ride_id").distinct().count()
print(f"Total de linhas: {total:,}")
print(f"ride_id distintos: {distintos:,}")
print(f"Duplicatas por ride_id: {total - distintos:,}")

### Regras de limpeza e cálculo do alvo

- **Alvo** `trip_duration_min` = diferença entre `ended_at` e `started_at`, em minutos.
- Remove viagens com **duração ≤ 0** (timestamps invertidos/errados).
- Remove viagens absurdamente longas: a Divvy cobra sobretaxa por bikes retidas; viagens acima de **3 horas (180 min)** são quase sempre bicicletas não devolvidas corretamente, não deslocamentos reais. Também removemos durações **< 1 min** (falsos inícios, docking imediato).
- Remove linhas sem **coordenadas de origem/destino** (necessárias para a distância).
- Remove **duplicatas** de `ride_id`.
- Mantém apenas valores válidos de `rideable_type` e `member_casual`.

In [ ]:
from pyspark.sql.functions import unix_timestamp, round as _round


def limpar(df):
    df = df.dropDuplicates(["ride_id"])
    df = df.withColumn(
        "trip_duration_min",
        _round((unix_timestamp("ended_at") - unix_timestamp("started_at")) / 60.0, 2),
    )
    return (df
        .filter(col("trip_duration_min") >= 1.0)
        .filter(col("trip_duration_min") <= 180.0)
        .filter(col("start_lat").isNotNull() & col("start_lng").isNotNull())
        .filter(col("end_lat").isNotNull() & col("end_lng").isNotNull())
        .filter(col("rideable_type").isNotNull())
        .filter(col("member_casual").isin(["member", "casual"]))
    )


limpo = limpar(bruto)
n_limpo = limpo.count()
print(f"Após limpeza: {n_limpo:,} viagens válidas "
      f"({n_limpo / total * 100:.1f}% do bruto) | removidas: {total - n_limpo:,}")

# Estatística descritiva do alvo depois da limpeza
limpo.select("trip_duration_min").describe().show()

### Amostragem declarada

O **conjunto original limpo** tem `n_limpo` viagens (mostrado acima). Para o experimento rodar rápido no Colab, extraímos uma **amostra aleatória simples** de `N_AMOSTRA` linhas, com semente fixa para reprodutibilidade. A amostra continua acima dos 200 mil registros exigidos.

In [ ]:
fracao = min(1.0, N_AMOSTRA * 1.1 / n_limpo)
amostra = limpo.sample(fraction=fracao, seed=SEED).limit(N_AMOSTRA)
amostra = amostra.cache()
print(f"Amostra de trabalho: {amostra.count():,} viagens "
      f"(de {n_limpo:,} válidas no conjunto original)")

## 4. Engenharia de atributos

A partir dos campos brutos criamos atributos com poder preditivo sobre a duração:

- **`dist_km`** — distância *haversine* (em km) entre a estação de origem e a de destino. É o principal preditor físico da duração.
- **`pickup_hour`** — hora do início (0–23): captura congestionamento e padrões de uso ao longo do dia.
- **`eh_fim_de_semana`** — passeios de fim de semana tendem a ser mais longos que deslocamentos de dias úteis.
- **`rideable_type`** — bicicleta elétrica costuma ser mais rápida (menor duração para a mesma distância).
- **`member_casual`** — usuários casuais (turistas) e membros (commuters) têm perfis de uso diferentes.

Justificativa das categóricas: `rideable_type` e `member_casual` têm poucos níveis e são nominais, então usamos `StringIndexer` + `OneHotEncoder`. As numéricas são padronizadas com `StandardScaler` (a regressão linear é sensível à escala dos atributos). Tudo é empacotado em um `Pipeline`, ajustado **apenas no treino** para evitar vazamento de dados.

In [ ]:
from pyspark.sql.functions import (hour, dayofweek, radians, sin, cos, asin,
                                   sqrt, pow as _pow, lit)


def criar_atributos(df):
    # Distância haversine (km), raio médio da Terra = 6371 km
    dlat = radians(col("end_lat") - col("start_lat"))
    dlng = radians(col("end_lng") - col("start_lng"))
    a = (_pow(sin(dlat / 2), 2)
         + cos(radians(col("start_lat"))) * cos(radians(col("end_lat")))
         * _pow(sin(dlng / 2), 2))
    dist = lit(2) * 6371.0 * asin(sqrt(a))
    return (df
        .withColumn("dist_km", _round(dist, 3))
        .withColumn("pickup_hour", hour("started_at").cast("double"))
        .withColumn("eh_fim_de_semana",
                    when(dayofweek("started_at").isin([1, 7]), 1.0).otherwise(0.0))
    )


COLUNAS_MODELO = ["dist_km", "pickup_hour", "eh_fim_de_semana",
                  "rideable_type", "member_casual"]

dados = criar_atributos(amostra).select(*COLUNAS_MODELO, "trip_duration_min")
dados = dados.filter(col("dist_km").isNotNull())
dados.show(8)
dados.select("dist_km", "pickup_hour", "trip_duration_min").describe().show()

### Divisão treino/teste

80/20 com semente fixa. O modelo e o pipeline de preparação serão ajustados **somente** no conjunto de treino.

In [ ]:
train_df, test_df = dados.randomSplit([0.8, 0.2], seed=SEED)
train_df = train_df.cache()
test_df = test_df.cache()
print(f"Treino: {train_df.count():,} | Teste: {test_df.count():,}")

### Pipeline de preparação

Estágios: `StringIndexer` → `OneHotEncoder` nas categóricas; `VectorAssembler` + `StandardScaler` nas numéricas; `VectorAssembler` final produzindo a coluna `features`.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer, OneHotEncoder, VectorAssembler,
                                StandardScaler)

num_cols = ["dist_km", "pickup_hour", "eh_fim_de_semana"]

idx_ride = StringIndexer(inputCol="rideable_type", outputCol="ride_idx",
                         handleInvalid="keep")
idx_user = StringIndexer(inputCol="member_casual", outputCol="user_idx",
                         handleInvalid="keep")
ohe = OneHotEncoder(inputCols=["ride_idx", "user_idx"],
                    outputCols=["ride_vec", "user_vec"])
assembler_num = VectorAssembler(inputCols=num_cols, outputCol="features_num")
scaler = StandardScaler(inputCol="features_num", outputCol="features_num_scaled",
                        withMean=True, withStd=True)
assembler_final = VectorAssembler(
    inputCols=["features_num_scaled", "ride_vec", "user_vec"], outputCol="features")

estagios_prep = [idx_ride, idx_user, ohe, assembler_num, scaler, assembler_final]

## 5. Modelagem — Regressão Linear com ajuste de hiperparâmetros

O estimador final é a `LinearRegression`. A preparação e o regressor ficam juntos em um único `Pipeline`, então a busca de hiperparâmetros re-ajusta toda a cadeia apenas com os dados de treino de cada *fold*/split — **sem vazamento**.

### Escolha do método de tuning: `TrainValidationSplit`

O Spark ML oferece duas estratégias de busca: `CrossValidator` (validação cruzada com *k folds*) e `TrainValidationSplit` (uma única divisão treino/validação).

Optamos por **`TrainValidationSplit`** porque:

- em **dados massivos**, cada avaliação de uma combinação de hiperparâmetros é cara; o `CrossValidator` com *k* folds treina o modelo *k* vezes por combinação, multiplicando o custo. Com **centenas de milhares de registros**, uma única divisão de validação já é estatisticamente estável — a variância entre folds é pequena porque cada partição é grande;
- é a escolha recomendada pela própria documentação do Spark quando o volume de dados é alto e o custo de treino domina, mantendo a mesma proteção contra vazamento;
- reduz o tempo total de execução no Colab de forma significativa, sem perda relevante de qualidade da seleção.

A grade varia a regularização (`regParam`) e o balanço L1/L2 (`elasticNetParam`) da regressão linear.

In [ ]:
import time
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder

lr = LinearRegression(featuresCol="features", labelCol="trip_duration_min")
pipeline = Pipeline(stages=estagios_prep + [lr])

grade = (ParamGridBuilder()
         .addGrid(lr.regParam, [0.0, 0.01, 0.1, 0.5])
         .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
         .build())

avaliador_rmse = RegressionEvaluator(labelCol="trip_duration_min",
                                     predictionCol="prediction",
                                     metricName="rmse")

tvs = TrainValidationSplit(estimator=pipeline,
                           estimatorParamMaps=grade,
                           evaluator=avaliador_rmse,
                           trainRatio=0.8,
                           parallelism=2,
                           seed=SEED)

inicio = time.time()
tvs_model = tvs.fit(train_df)
print(f"Busca concluída em {time.time() - inicio:.1f}s")

melhor_pipeline = tvs_model.bestModel
melhor_lr = melhor_pipeline.stages[-1]
print(f"Melhores hiperparâmetros: regParam={melhor_lr.getRegParam()}, "
      f"elasticNetParam={melhor_lr.getElasticNetParam()}")

## 6. Avaliação e interpretação

Avaliamos o **melhor modelo** no conjunto de **teste** (nunca visto no treino nem na validação), com as três métricas exigidas: **RMSE**, **MAE** e **R²**.

### Comparação com um baseline

A referência é um **modelo trivial** que prevê sempre a **duração média do treino**. Qualquer modelo útil precisa superá-lo com folga; a comparação mostra o quanto os atributos realmente explicam a duração.

In [ ]:
pred = melhor_pipeline.transform(test_df)

metricas = {m: RegressionEvaluator(labelCol="trip_duration_min",
                                   predictionCol="prediction",
                                   metricName=m).evaluate(pred)
            for m in ["rmse", "mae", "r2"]}

# Baseline: prever sempre a média do treino
from pyspark.sql.functions import avg
media_treino = train_df.select(avg("trip_duration_min")).first()[0]
base_pred = test_df.withColumn("prediction", lit(media_treino))
base = {m: RegressionEvaluator(labelCol="trip_duration_min",
                               predictionCol="prediction",
                               metricName=m).evaluate(base_pred)
        for m in ["rmse", "mae", "r2"]}

print(f"Duração média no treino (baseline): {media_treino:.2f} min\n")
print(f"{'Métrica':<8}{'Baseline (média)':>20}{'LinearRegression':>20}")
for m in ["rmse", "mae", "r2"]:
    print(f"{m.upper():<8}{base[m]:>20.3f}{metricas[m]:>20.3f}")

### Coeficientes do modelo

Como os atributos numéricos foram padronizados, a magnitude dos coeficientes indica a importância relativa de cada um na previsão da duração.

In [ ]:
import pandas as pd

# nomes dos atributos na ordem do VectorAssembler final:
# [dist_km, pickup_hour, eh_fim_de_semana (escalados)] + one-hot de ride e user
nomes = num_cols + ["rideable_type(OHE)", "member_casual(OHE)"]
coefs = list(melhor_lr.coefficients)
# os blocos one-hot podem ter mais de uma posição; resumimos pelo primeiro peso de cada
print("Intercepto:", round(melhor_lr.intercept, 3))
print("\nCoeficientes (atributos numéricos padronizados):")
for nome, c in zip(num_cols, coefs[:len(num_cols)]):
    print(f"  {nome:<18} {c:>8.3f}")
print("\nDemais coeficientes (blocos one-hot):")
for c in coefs[len(num_cols):]:
    print(f"  {c:>8.3f}")

### Interpretação dos resultados

- **A regressão linear supera o baseline** em RMSE, MAE e R² (ver tabela). Enquanto o baseline tem R² ≈ 0 (por definição, prever a média não explica variância), o modelo alcança um R² positivo, confirmando que os atributos carregam informação sobre a duração.
- **`dist_km` é o preditor dominante**, com o maior coeficiente em módulo: quanto mais distantes a origem e o destino, mais longa a viagem — o resultado físico esperado.
- **`member_casual` e `rideable_type` deslocam a duração**: usuários casuais tendem a viagens mais longas (uso recreativo) e bicicletas elétricas a viagens mais curtas para a mesma distância (maior velocidade).
- **`pickup_hour` e `eh_fim_de_semana`** têm efeito menor, mas capturam variação por horário de pico e por perfil de uso no fim de semana.

### Limitações

- A **distância haversine** é a linha reta entre estações, não o trajeto real percorrido (que é maior e varia com a malha viária). Isso impõe um teto ao R² de um modelo linear.
- A relação entre distância e duração **não é perfeitamente linear** (trânsito, paradas, desvios); um modelo não linear (árvores, GBT) provavelmente capturaria melhor essas interações — fica como melhoria futura.
- Viagens de ida e volta à mesma estação têm `dist_km` ≈ 0 mas duração positiva, adicionando ruído.
- O modelo foi treinado em **3 meses de 2024 (primavera/início de verão)**; sazonalidade (inverno de Chicago) pode mudar os padrões, limitando a generalização para o ano todo.

## 7. Persistência do modelo

O pipeline completo (preparação + regressão linear ajustada) é salvo com `save()` do Spark ML. Ele pode ser recarregado com `PipelineModel.load()` e aplicado diretamente a dados novos, sem refazer a preparação à mão.

In [ ]:
from pyspark.ml import PipelineModel

melhor_pipeline.write().overwrite().save(DIR_MODELO)
print(f"Modelo salvo em: {DIR_MODELO}")

# Verificação: recarrega e prevê 5 linhas do teste
carregado = PipelineModel.load(DIR_MODELO)
carregado.transform(test_df) \
    .select("dist_km", "rideable_type", "member_casual",
            "trip_duration_min", "prediction") \
    .show(5, truncate=False)

## 8. Conclusão

Construímos, de ponta a ponta e inteiramente em **PySpark**, um modelo de regressão para prever a **duração das viagens** do sistema Divvy de Chicago — uma base pública, com mais de 1,6 milhão de viagens nos três meses usados e justificada como um problema de dados massivos (volume anual na casa dos milhões e ingestão contínua).

O fluxo cobriu: ingestão com **schema manual (`StructType`)**; exploração e limpeza no Spark (estatísticas descritivas, nulos, duplicatas e remoção de durações inválidas); engenharia de atributos em um **`Pipeline`** (distância haversine, hora, fim de semana, codificação one-hot e padronização); modelagem com **`LinearRegression`** e ajuste de hiperparâmetros via **`TrainValidationSplit`** sem vazamento de dados; avaliação com **RMSE, MAE e R²** comparados a um **baseline** da média; e persistência com **`save()`**.

**Resultados:** o modelo supera o baseline nas três métricas, com a distância entre estações como principal preditor da duração.

**Possíveis melhorias:**

- substituir a distância em linha reta pela **distância de rota real** (API de roteamento) ou adicionar atributos de malha viária;
- testar **modelos não lineares** (Random Forest, GBT) para capturar interações entre distância, horário e tipo de bike;
- incorporar **sazonalidade** (meses de inverno) e variáveis externas (clima, eventos) para melhorar a generalização ao ano inteiro;
- escalar o treino para a **base histórica completa** em um cluster, trocando apenas o caminho de leitura e o `master`.